## Simple GenAI APP uisng Langchain

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]=os.getenv("LANGSMITH_TRACING")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_ENDPOINT"]=os.getenv("LANGCHAIN_ENDPOINT")

In [ ]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")
print(llm)

In [ ]:
## Data Ingestion-- From the website we need to scrape data

from langchain_community.document_loaders import WebBaseLoader
loader=WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
loader

d:\2026-courses\agenticai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
docs= loader.load()
docs

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUs

In [6]:
### Laod Data--> Docs--> Devide our text into chunks --> Embed the chunks --> Store in a vector database

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
document_chunks= text_splitter.split_documents(docs)

In [7]:
document_chunks

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUs

In [ ]:
# Intialize the embeddings model
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [ ]:
# Build Vector Store and store the embneddings in it
from langchain_community.vectorstores import FAISS

vectorstoredb=FAISS.from_documents(document_chunks,embeddings)
vectorstoredb

In [ ]:
# Apply similarity search now
query="LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications."
result=vectorstoredb.similarity_search(query)
result[0].page_content

In [ ]:
## Retreiver chain, Document Chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
        Answer the question based on the context provided.
        <context>
        {context}
        </context>
    """
)



In [ ]:
document_chain = create_stuff_documents_chain(llm,prompt)
    

In [ ]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [ ]:
### Retriever: Convert the vector store into a retriever to get the most relevant documents
from langchain.chains import create_retrieval_chain

retriever = vectorstoredb.as_retriever()
# Create the retrieval chain that accepts two inputs:
# 1. The question
# 2. The context
# It will return a dictionary with two keys:
# 1. answer: The answer to the question
# 2. context: The context used to answer the question
# document_chain is the chain that will be used to answer the question
# retriever is the retriever that will be used to get the most relevant documents
# create_retrieval_chain will create a chain that will use the retriever to get the most relevant documents and then use the document_chain to answer the question
retriever_chain= create_retrieval_chain(retriever,document_chain) 




In [ ]:
## Get the response from the llm
response= retriever_chain(
    {
        "input":"LangSmith has two usage limits: total traces and extended",
    }
)

response['answer']